# Chapter 10: Clustering Techniques

## Ringkasan Bab & Penjelasan Teoretis
**Clustering (Klasterisasi)** adalah metode *unsupervised learning* yang bertujuan menemukan pengelompokan alami (*natural groupings*) dari titik-titik data berdasarkan kemiripan karakteristiknya tanpa bantuan label target.

### 1. K-Means, Hierarchical Clustering, DBSCAN, dan GMM
* **K-Means Clustering (`KMeans`)**: Mempartisi data ke dalam $k$ klaster dengan meminimalkan jarak kuadrat setiap titik terhadap titik pusat klasternya (*centroid* / *inertia*). Bekerja sangat baik pada klaster berbentuk bola (*isotropic/convex*).
* **Hierarchical / Agglomerative Clustering (`AgglomerativeClustering`)**: Membangun hierarki klaster dari bawah ke atas (*bottom-up*) dengan menggabungkan pasangan klaster terdekat secara bertahap berdasarkan kriteria *linkage* (`ward`, `complete`, `average`).
* **DBSCAN (`DBSCAN`)**: *Density-Based Spatial Clustering of Applications with Noise* mengelompokkan titik-titik yang memiliki kepadatan tinggi (minimal `min_samples` dalam radius `eps`) dan menandai titik terisolasi sebagai *outlier/noise* ($-1$). Mampu mengenali klaster berbentuk non-linear seperti bulan sabit (*moons*).
* **Gaussian Mixture Models (`GaussianMixture`)**: Model probabilistik yang mengasumsikan data dihasilkan dari campuran sejumlah distribusi Gaussian berdimensi banyak.

### 2. Metrik Evaluasi Klaster
* **Silhouette Score**: Mengukur seberapa dekat suatu titik dengan klasternya sendiri dibandingkan dengan klaster tetangga terdekat (rentang $[-1, 1]$, semakin mendekati $1$ semakin terpisah dengan baik).
* **Adjusted Rand Index (ARI)**: Mengukur kesamaan antara hasil klasterisasi dengan label referensi (jika tersedia untuk validasi).

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.datasets import load_iris, make_moons
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans, AgglomerativeClustering, DBSCAN
from sklearn.mixture import GaussianMixture
from sklearn.decomposition import PCA
from sklearn.metrics import silhouette_score, adjusted_rand_score

# 1. K-Means dan Agglomerative Clustering pada Dataset Iris
iris = load_iris()
X_iris = StandardScaler().fit_transform(iris.data)

kmeans = KMeans(n_clusters=3, random_state=2024, n_init=10)
labels_km = kmeans.fit_predict(X_iris)

agg = AgglomerativeClustering(n_clusters=3, linkage='ward')
labels_agg = agg.fit_predict(X_iris)

gmm = GaussianMixture(n_components=3, random_state=2024)
labels_gmm = gmm.fit_predict(X_iris)

eval_df = pd.DataFrame({
    'Algorithm': ['K-Means', 'Agglomerative (Ward)', 'Gaussian Mixture Model (GMM)'],
    'Silhouette Score': [
        silhouette_score(X_iris, labels_km),
        silhouette_score(X_iris, labels_agg),
        silhouette_score(X_iris, labels_gmm)
    ],
    'Adjusted Rand Index (vs True Iris Class)': [
        adjusted_rand_score(iris.target, labels_km),
        adjusted_rand_score(iris.target, labels_agg),
        adjusted_rand_score(iris.target, labels_gmm)
    ]
})
display(eval_df)

# Visualisasi Hasil Klasterisasi Iris dengan Proyeksi PCA 2D
X_pca = PCA(n_components=2).fit_transform(X_iris)
fig, axes = plt.subplots(1, 3, figsize=(16, 5))
for ax, lbl, title in zip(axes, [labels_km, labels_agg, labels_gmm], ['K-Means', 'Agglomerative', 'GMM']):
    ax.scatter(X_pca[:, 0], X_pca[:, 1], c=lbl, cmap='viridis', edgecolor='k', s=40)
    ax.set_title(f'{title} on Iris (PCA Projection)')
    ax.set_xlabel('PC1')
    ax.set_ylabel('PC2')
plt.tight_layout()
plt.show()

# 2. Perbandingan DBSCAN vs K-Means pada Data Non-Linear (Moons Dataset)
X_moons, y_moons = make_moons(n_samples=300, noise=0.07, random_state=2024)
X_moons_scaled = StandardScaler().fit_transform(X_moons)

km_moons = KMeans(n_clusters=2, random_state=2024, n_init=10).fit_predict(X_moons_scaled)
dbscan_moons = DBSCAN(eps=0.3, min_samples=5).fit_predict(X_moons_scaled)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 5))
ax1.scatter(X_moons_scaled[:, 0], X_moons_scaled[:, 1], c=km_moons, cmap='coolwarm', edgecolor='k')
ax1.set_title('K-Means on Moon Data (Linear Boundary Limitation)')

ax2.scatter(X_moons_scaled[:, 0], X_moons_scaled[:, 1], c=dbscan_moons, cmap='coolwarm', edgecolor='k')
ax2.set_title('DBSCAN on Moon Data (Captures Density Structure)')
plt.tight_layout()
plt.show()